# Compute & Store CLAP Audio Embeddings

**Run this notebook once** (or again whenever you add new songs) to build the embedding index used by `playlist_from_prompt.ipynb`.

- Scans `SONGS_DIR` for audio files
- Skips tracks already present in `data/embeddings.npz` (incremental)
- Saves/merges results to `data/embeddings.npz`

In [8]:
import sys
from pathlib import Path

import numpy as np
import torch

# Make utils importable when the notebook is run from the repo root
if "" not in sys.path:
    sys.path.insert(0, "")

import utils

In [9]:
# ── Configuration ─────────────────────────────────────────────────────────────
SONGS_DIR       = Path("data/fma_small")   # root folder that contains the MP3s
EMB_OUT         = Path("data/embeddings.npz")

CHECKPOINT_URL  = utils.DEFAULT_CHECKPOINT_URL
CHECKPOINT_PATH = utils.DEFAULT_CHECKPOINT_PATH

BATCH_SIZE      = 8
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Songs dir : {SONGS_DIR}")
print(f"Output    : {EMB_OUT}")
print(f"Device    : {DEVICE}")

Songs dir : data/fma_small
Output    : data/embeddings.npz
Device    : cuda


In [10]:
# ── Discover audio files ──────────────────────────────────────────────────────
all_files = utils.collect_audio_files(SONGS_DIR)
print(f"Found {len(all_files)} audio file(s) under {SONGS_DIR}")

Found 8000 audio file(s) under data/fma_small


In [11]:
# ── Load existing embeddings (incremental support) ────────────────────────────
if EMB_OUT.exists():
    known_paths, known_embs = utils.load_embeddings(EMB_OUT)
    known_set = set(known_paths)
else:
    known_paths, known_embs = [], np.empty((0, 0), dtype=np.float32)
    known_set = set()
    print("No existing embeddings found — computing from scratch.")

new_files = [p for p in all_files if str(p) not in known_set]
print(f"Already embedded : {len(known_paths)}")
print(f"New to embed     : {len(new_files)}")

No existing embeddings found — computing from scratch.
Already embedded : 0
New to embed     : 8000


In [12]:
# ── Load CLAP model (only if there are new files to embed) ────────────────────
if new_files:
    model = utils.load_clap_model(CHECKPOINT_PATH, CHECKPOINT_URL, device=DEVICE)
else:
    print("Nothing new to embed — skipping model loading.")
    model = None

Checkpoint already present: checkpoints/music_audioset_epoch_15_esc_90.14.pt


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load the specified checkpoint checkpoints/music_audioset_epoch_15_esc_90.14.pt from users.
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.bl

In [13]:
# ── Batch-compute embeddings for new files ────────────────────────────────────
# Files are processed one-by-one so a single corrupt MP3 only skips itself.
# A batch of size > 1 would crash the whole batch on one bad file.
new_embs_chunks = []
new_good_paths  = []
skipped         = []

if model is not None and new_files:
    new_paths_str = [str(p) for p in new_files]
    total = len(new_paths_str)

    for i, path in enumerate(new_paths_str):
        pct = (i + 1) / total * 100
        print(f"\r  Embedding {i + 1}/{total}  ({pct:.0f}%)", end="", flush=True)
        try:
            emb = model.get_audio_embedding_from_filelist(x=[path], use_tensor=False)
            emb = np.asarray(emb, dtype=np.float32)
            if emb.shape[0] == 0:
                raise ValueError("Empty embedding returned")
            new_embs_chunks.append(emb)
            new_good_paths.append(path)
        except Exception as e:
            skipped.append(path)
            print(f"\n  [SKIP] {Path(path).name}  — {type(e).__name__}: {e}")

    print()  # newline after progress bar

    if skipped:
        print(f"\nSkipped {len(skipped)} file(s) due to load errors.")

    if new_embs_chunks:
        new_embs = utils.l2_normalize(np.vstack(new_embs_chunks))
        print(f"New embeddings shape: {new_embs.shape}")
    else:
        print("No new embeddings produced.")

  Embedding 494/8000  (6%)

[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!


  Embedding 905/8000  (11%)

[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!


  Embedding 1185/8000  (15%)

...

  Embedding 4904/8000  (61%)
  [SKIP] 108925.mp3  — NoBackendError: 
  Embedding 4907/8000  (61%)

[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...


  Embedding 6966/8000  (87%)
  [SKIP] 133297.mp3  — NoBackendError: 
  Embedding 6969/8000  (87%)

[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...


  Embedding 8000/8000  (100%)

Skipped 6 file(s) due to load errors.
New embeddings shape: (7994, 512)


In [14]:
# ── Merge old + new and save ──────────────────────────────────────────────────
if new_embs_chunks:
    if known_paths:
        all_paths = known_paths + new_good_paths
        all_embs  = np.vstack([known_embs, new_embs])
    else:
        all_paths = new_good_paths
        all_embs  = new_embs

    utils.save_embeddings(EMB_OUT, all_paths, all_embs)
else:
    all_paths = known_paths
    all_embs  = known_embs
    print("Embeddings already up to date — nothing saved.")

Saved 7994 embeddings → data/embeddings.npz


In [15]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 55)
print(f"Total tracks indexed : {len(all_paths)}")
print(f"Embedding matrix     : {all_embs.shape}")
print(f"Stored at            : {EMB_OUT.resolve()}")
print()
print("Sample paths:")
for p in all_paths[:5]:
    print(f"  {p}")
if len(all_paths) > 5:
    print(f"  … and {len(all_paths) - 5} more")
print("=" * 55)

Total tracks indexed : 7994
Embedding matrix     : (7994, 512)
Stored at            : /home/nathan/Documents/PlaylistGenerator/data/embeddings.npz

Sample paths:
  data/fma_small/000/000002.mp3
  data/fma_small/000/000005.mp3
  data/fma_small/000/000010.mp3
  data/fma_small/000/000140.mp3
  data/fma_small/000/000141.mp3
  … and 7989 more
